# 评估示例：推送通知摘要提示词的回归检测

评估（Evals）是**面向任务**且需要反复迭代的，它是检查 LLM 集成运行状况并加以改进的最佳方式。

在下面的评估中，我们将聚焦于**检测提示词改动是否造成回归**这一任务。

我们的用例是：
1. 我有一个 LLM 集成，它接收一组推送通知，并将它们概括为一条精简的陈述。
2. 我想检测提示词的改动是否会导致行为回归

## 评估结构

评估包含两个部分：“Eval”和“Run”。“Eval”保存测试标准的配置以及“Run”的数据结构。一个 Eval 可以有多个 Run，这些 Run 都会依据你的测试标准进行评估。

In [1]:
import openai
from openai.types.chat import ChatCompletion
import pydantic
import os

os.environ["OPENAI_API_KEY"] = os.environ.get("OPENAI_API_KEY", "your-api-key")

## 用例

我们要测试的是下面这个集成：推送通知摘要。它接收多条推送通知并将其合并为一条，这是一次 chat completions 调用。

In [ ]:
class PushNotifications(pydantic.BaseModel):
    notifications: str

print(PushNotifications.model_json_schema())

In [ ]:
DEVELOPER_PROMPT = """
You are a helpful assistant that summarizes push notifications.
You are given a list of push notifications and you need to collapse them into a single one.
Output only the final summary, nothing else.
"""

def summarize_push_notification(push_notifications: str) -> ChatCompletion:
    result = openai.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {"role": "developer", "content": DEVELOPER_PROMPT},
            {"role": "user", "content": push_notifications},
        ],
    )
    return result

example_push_notifications_list = PushNotifications(notifications="""
- Alert: Unauthorized login attempt detected.
- New comment on your blog post: "Great insights!"
- Tonight's dinner recipe: Pasta Primavera.
""")
result = summarize_push_notification(example_push_notifications_list.notifications)
print(result.choices[0].message.content)

# 设置评估

Eval 保存在多个 *Run* 之间共享的配置，它包含两个组成部分：
1. 数据源配置 `data_source_config`——后续 *Run* 需要遵循的模式（列）。
    - `data_source_config` 使用 JSON Schema 来定义 Eval 中可用的变量。
2. 测试标准 `testing_criteria`——针对数据源中的每一*行*，你如何判断集成是否正常工作。

在这个用例中，我们想测试推送通知摘要的补全结果是否良好，因此会据此设置评估。

In [4]:
# We want our input data to be available in our variables, so we set the item_schema to
# PushNotifications.model_json_schema()
data_source_config = {
    "type": "custom",
    "item_schema": PushNotifications.model_json_schema(),
    # We're going to be uploading completions from the API, so we tell the Eval to expect this
    "include_sample_schema": True,
}

这个 data_source_config 定义了整个评估中可用的变量。

这个 item 模式：
```json
{
  "properties": {
    "notifications": {
      "title": "Notifications",
      "type": "string"
    }
  },
  "required": ["notifications"],
  "title": "PushNotifications",
  "type": "object"
}
```
意味着我们可以在评估中使用变量 `{{item.notifications}}`。

`"include_sample_schema": True`
意味着我们可以在评估中使用变量 `{{sample.output_text}}`。

**接下来，我们将使用这些变量来设置测试标准。**

In [5]:
GRADER_DEVELOPER_PROMPT = """
Label the following push notification summary as either correct or incorrect.
The push notification and the summary will be provided below.
A good push notificiation summary is concise and snappy.
If it is good, then label it as correct, if not, then incorrect.
"""
GRADER_TEMPLATE_PROMPT = """
Push notifications: {{item.notifications}}
Summary: {{sample.output_text}}
"""
push_notification_grader = {
    "name": "Push Notification Summary Grader",
    "type": "label_model",
    "model": "o3-mini",
    "input": [
        {
            "role": "developer",
            "content": GRADER_DEVELOPER_PROMPT,
        },
        {
            "role": "user",
            "content": GRADER_TEMPLATE_PROMPT,
        },
    ],
    "passing_labels": ["correct"],
    "labels": ["correct", "incorrect"],
}

`push_notification_grader` 是一个模型评分器（LLM 作为评判者），它查看输入 `{{item.notifications}}` 和生成的摘要 `{{sample.output_text}}`，并将其标记为 "correct" 或 "incorrect"。
然后，我们通过 "passing_labels" 指定什么样的答案算作通过。

注意：在底层，这里使用了结构化输出，以确保标签始终有效。

**现在我们来创建评估，并开始向其中添加数据！**

In [6]:
eval_create_result = openai.evals.create(
    name="Push Notification Summary Workflow",
    metadata={
        "description": "This eval checks if the push notification summary is correct.",
    },
    data_source_config=data_source_config,
    testing_criteria=[push_notification_grader],
)

eval_id = eval_create_result.id

# 创建 Run

现在我们已经用 test_criteria 设置好了评估，可以开始添加一系列 Run 了！
我们先从一些推送通知数据开始。

In [7]:
push_notification_data = [
        """
- New message from Sarah: "Can you call me later?"
- Your package has been delivered!
- Flash sale: 20% off electronics for the next 2 hours!
""",
        """
- Weather alert: Thunderstorm expected in your area.
- Reminder: Doctor's appointment at 3 PM.
- John liked your photo on Instagram.
""",
        """
- Breaking News: Local elections results are in.
- Your daily workout summary is ready.
- Check out your weekly screen time report.
""",
        """
- Your ride is arriving in 2 minutes.
- Grocery order has been shipped.
- Don't miss the season finale of your favorite show tonight!
""",
        """
- Event reminder: Concert starts at 7 PM.
- Your favorite team just scored!
- Flashback: Memories from 3 years ago.
""",
        """
- Low battery alert: Charge your device.
- Your friend Mike is nearby.
- New episode of "The Tech Hour" podcast is live!
""",
        """
- System update available.
- Monthly billing statement is ready.
- Your next meeting starts in 15 minutes.
""",
        """
- Alert: Unauthorized login attempt detected.
- New comment on your blog post: "Great insights!"
- Tonight's dinner recipe: Pasta Primavera.
""",
        """
- Special offer: Free coffee with any breakfast order.
- Your flight has been delayed by 30 minutes.
- New movie release: "Adventures Beyond" now streaming.
""",
        """
- Traffic alert: Accident reported on Main Street.
- Package out for delivery: Expected by 5 PM.
- New friend suggestion: Connect with Emma.
"""]

第一个 Run 将使用上面的补全函数 `summarize_push_notification` 作为默认基线。
我们将遍历数据集，进行补全调用，然后将结果作为一个 Run 提交评分。

In [ ]:
run_data = []
for push_notifications in push_notification_data:
    result = summarize_push_notification(push_notifications)
    run_data.append({
        "item": PushNotifications(notifications=push_notifications).model_dump(),
        "sample": result.model_dump()
    })

eval_run_result = openai.evals.runs.create(
    eval_id=eval_id,
    name="baseline-run",
    data_source={
        "type": "jsonl",
        "source": {
            "type": "file_content",
            "content": run_data,
        }
    },
)
print(eval_run_result)
# Check out the results in the UI
print(eval_run_result.report_url)

现在我们来模拟一次回归。下面是原始提示词，我们来模拟开发者把提示词改坏的情况。

```python
DEVELOPER_PROMPT = """
You are a helpful assistant that summarizes push notifications.
You are given a list of push notifications and you need to collapse them into a single one.
Output only the final summary, nothing else.
"""
```

In [9]:
DEVELOPER_PROMPT = """
You are a helpful assistant that summarizes push notifications.
You are given a list of push notifications and you need to collapse them into a single one.
You should make the summary longer than it needs to be and include more information than is necessary.
"""

def summarize_push_notification_bad(push_notifications: str) -> ChatCompletion:
    result = openai.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {"role": "developer", "content": DEVELOPER_PROMPT},
            {"role": "user", "content": push_notifications},
        ],
    )
    return result

In [ ]:
run_data = []
for push_notifications in push_notification_data:
    result = summarize_push_notification_bad(push_notifications)
    run_data.append({
        "item": PushNotifications(notifications=push_notifications).model_dump(),
        "sample": result.model_dump()
    })

eval_run_result = openai.evals.runs.create(
    eval_id=eval_id,
    name="regression-run",
    data_source={
        "type": "jsonl",
        "source": {
            "type": "file_content",
            "content": run_data,
        }
    },
)
print(eval_run_result.report_url)

查看该报告，你会发现它的得分远低于基线 Run。

## 恭喜，你刚刚阻止了一个 bug 被发布给用户

补充说明：
Evals 目前尚未原生支持 `responses` API，不过你可以使用以下代码将其转换为 `completions` 格式。

In [ ]:
def summarize_push_notification_responses(push_notifications: str):
    result = openai.responses.create(
                model="gpt-4o",
                input=[
                    {"role": "developer", "content": DEVELOPER_PROMPT},
                    {"role": "user", "content": push_notifications},
                ],
            )
    return result
def transform_response_to_completion(response):
    completion = {
        "model": response.model,
        "choices": [{
        "index": 0,
        "message": {
            "role": "assistant",
            "content": response.output_text
        },
        "finish_reason": "stop",
    }]
    }
    return completion

run_data = []
for push_notifications in push_notification_data:
    response = summarize_push_notification_responses(push_notifications)
    completion = transform_response_to_completion(response)
    run_data.append({
        "item": PushNotifications(notifications=push_notifications).model_dump(),
        "sample": completion
    })

report_response = openai.evals.runs.create(
    eval_id=eval_id,
    name="responses-run",
    data_source={
        "type": "jsonl",
        "source": {
            "type": "file_content",
            "content": run_data,
        }
    },
)
print(report_response.report_url)